## Add required fields Tasks for point, line, polygon layers

This notebook updates the schema of an existing point, line or polygon feature layer, preparing it to support the Tasks functionality in ArcGIS Field Maps.

The notebook performs the following actions:

- Adds a series of fields and necessary domains
- Enables attachments if needed 

Note: you must be the owner of the feature layer or an organization admin user

### Steps

0. Identify a hosted feature layer that you want to enable with tasks.  Edit tracking and sync should be enabled.

1. Update the following variables

    - `item_id`
    - `feature_layer_index`

2. Run all cells


Reference:
Tasks Information Model: https://doc.arcgis.com/en/field-maps/latest/prepare-maps/prepare-tasks.htm#ESRI_SECTION1_3BBDB65A5B75485D89FC4BE0F92A32BC


In [ ]:
# Import libraries
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import uuid

In [ ]:
# Sign in to organization
PORTAL_URL = 'https://www.arcgis.com'   # ArcGIS Online, or your Enterprise URL
username   = input('Enter username: ')

gis = GIS(PORTAL_URL, username)
# gis = GIS("home") # use this method when running as a hosted notebook in AGOL/Enterprise

print(f'Connected as: {gis.properties.user.username}')

In [ ]:
# update this information prior to running
ITEM_ID = '' # item_id of feature layer
FEATURE_LAYER_INDEX = 0 # index of layer you want to update

# get the feature layer by ID
item = gis.content.get(ITEM_ID)

if (item is None):
    raise TypeError('Cannot Find item')

# check the item type is a feature service
if (item.type != 'Feature Service'):
    raise TypeError('Item is not a feature service')

feature_layer_collection = FeatureLayerCollection.fromitem(item)

feature_layer = feature_layer_collection.layers[FEATURE_LAYER_INDEX]

# get the geometry type
geometry_type = feature_layer.properties.geometryType

# make sure layer is a point, line, or polygon layer
if ((geometry_type != 'esriGeometryPolyline') and 
    (geometry_type != 'esriGeometryPolygon')and 
    (geometry_type != 'esriGeometryPoint')):
    raise TypeError('Feature layer is not a point, line, or polygon layer')

**Workforce Assigment Types (Optional)**
If your task layer originated from a workforce project, especially if you ran the workforce_flat_service notebook to create it, you may want to bring your Assignment Type values from Workforce and apply them to the domain of the esritask_type field. Use the next cell to pass in your Workforce feature service ID and create the function to gather the domain values.

In [ ]:
WORKFORCE_FS_ID = '' # enter the item ID of your workforce assignment table

wf_item = gis.content.get(WORKFORCE_FS_ID)

flc = FeatureLayerCollection.fromitem(wf_item)

# check the index of the assignment type table
if flc.tables[1].properties.name != 'Assignment Types':
    print('Please enter the table index of your assignment type table')
    ASSIGN_TABLE_INDEX = input('Enter table index:')
else:
    ASSIGN_TABLE_INDEX = 1

# query the assignment type table for unique descriptions
assignment_types_query = flc.tables[ASSIGN_TABLE_INDEX].query(out_fields='description', return_distinct_values=True)
assignment_types = [f.attributes['description'] for f in assignment_types_query]

if not assignment_types:
    print('No assignment types found')
else:
    print('All assignment types from Workforce project:')
    print(assignment_types)

cv_list = []
cv_code = 0
for t in assignment_types:
    cvd_dict = {}
    cvd_dict['name'] = t
    cvd_dict['code'] = cv_code
    cv_code += 1
    cv_list.append(cvd_dict)

In [ ]:
# Ensure all fields are present; if not, add them to the definition

# Check for workforce project-derived values
try:
    cv_list
except:
    cv_list = []

if cv_list:
    domain_codes = cv_list

else:
    # these names can be updated with your preferred task type domain values
    domain_codes = [{'name': 'Python Added Fields 1', 'code': 0},
                    {'name': 'Python Added Fields 2', 'code': 1}]

# new fields which need to be added
tasks_fields = {'fields': []}

# Get existing fields from feature service and create lowercase field list
feature_layer_fields = feature_layer.properties.fields
lower_fields = [field['name'].lower() for field in feature_layer_fields]

# esritask_type
if 'esritask_type' not in lower_fields:
    tasks_fields['fields'].append({'name': 'esritask_type',
                                            'type': 'esriFieldTypeInteger',
                                            'alias': 'Task Type',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True,
                                            'domain': {'type': 'codedValue',
                                                    'name': 'ESRITASK_TYPE_DOMAIN_' + str(uuid.uuid4()),
                                                    'codedValues': domain_codes},
                                            'defaultValue': 0})

# esritask_status
esritask_status_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_status']

if not esritask_status_field:
    tasks_fields['fields'].append({'name': 'esritask_status',
                                            'type': 'esriFieldTypeInteger',
                                            'alias': 'Status',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True,
                                            'domain':  {'type': 'codedValue',
                                                    'name': 'ESRITASK_STATUS_DOMAIN_' + str(uuid.uuid4()),
                                                    'codedValues': [
                                                        {'name': 'Unassigned', 'code': 0},
                                                        {'name': 'Assigned', 'code': 1},
                                                        {'name': 'In Progress', 'code': 2},
                                                        {'name': 'Completed', 'code': 3}]},
                                            'defaultValue': 0})

# esritask_assignee
esritask_assignee_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_assignee']

if not esritask_assignee_field:
    tasks_fields['fields'].append({'name': 'esritask_assignee',
                                            'type': 'esriFieldTypeString',
                                            'alias': 'Assignee',
                                            'sqlType': 'sqlTypeOther',
                                            'length': 255,
                                            'nullable': True,
                                            'editable': True,
                                            'domain':  {'type': 'codedValue',
                                                    'name': 'ESRITASK_ASSIGNEE_DOMAIN_' + str(uuid.uuid4()),
                                                    'codedValues': [
                                                        {'name': 'Assignee name 1', 'code': 'assignee_username_1'},
                                                        {'name': 'Assignee name 2', 'code': 'assignee_username_2'}]},
                                            'defaultValue': None})

# esritask_priority
esritask_priority_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_priority']

if not esritask_priority_field:
    tasks_fields['fields'].append({'name': 'esritask_priority',
                                            'type': 'esriFieldTypeInteger',
                                            'alias': 'Priority',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True,
                                            'domain':  {'type': 'codedValue',
                                                    'name': 'ESRITASK_PRIORITY_DOMAIN_' + str(uuid.uuid4()),
                                                    'codedValues': [
                                                        {'name': 'None', 'code': 0},
                                                        {'name': 'Low', 'code': 1},
                                                        {'name': 'Medium', 'code': 2},
                                                        {'name': 'High', 'code': 3},
                                                        {'name': 'Critical', 'code': 4}]},
                                            'defaultValue': 0})

# esritask_duedate
esritask_duedate_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_duedate']

if not esritask_duedate_field:
    tasks_fields['fields'].append({'name': 'esritask_duedate',
                                            'type': 'esriFieldTypeDate',
                                            'alias': 'Due Date',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True})

# esritask_description
esritask_description_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_description']

if not esritask_description_field:
    tasks_fields['fields'].append({'name': 'esritask_description',
                                            'type': 'esriFieldTypeString',
                                            'alias': 'Description',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True,
                                            'length':4000})

# esritask_notes
esritask_notes_field = [field for field in feature_layer_fields if field['name'].lower() == 'esritask_notes']

if not esritask_notes_field:
    tasks_fields['fields'].append({'name': 'esritask_notes',
                                            'type': 'esriFieldTypeString',
                                            'alias': 'Notes',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': True,
                                            'editable': True,
                                            'length':4000})

# GlobalID
globalid_field = [field for field in feature_layer_fields if field['name'].lower() == 'globalid']

if not globalid_field:
    tasks_fields['fields'].append({'name': 'GlobalID',
                                            'type': 'esriFieldTypeGlobalID',
                                            'alias': 'GlobalID',
                                            'sqlType': 'sqlTypeOther',
                                            'nullable': False,
                                            'editable': False,
                                            'defaultValue': "NEWID() WITH VALUES"})

In [ ]:
# Check if AddToDefinition operation needs to be added.
initial_featurelayer_fields_count = len(feature_layer_fields)

operations = [ ] 

if (len(tasks_fields['fields']) + initial_featurelayer_fields_count ) > initial_featurelayer_fields_count:
    operations.append('addToDefinition')
    
# Check if any changes are required
if not operations:
    print('No changes to tasks fields.')
else:
    # Add or update service definition
    for operation in operations:

        # Add
        if operation == 'addToDefinition':
            response = feature_layer.manager.add_to_definition(tasks_fields)
            print('Successfully added Tasks fields.')

        # Modify
        else:
            response = feature_layer.manager.update_definition(tasks_fields)
            print('Successfully updated Tasks fields.')

        result = response['success']

        if not result:
            print('Failed to update Feature layer service definition.')
        else:
            print('Service definition updated successfully.')